In [17]:
## Import divers
import pandas as pd
import ast
from sklearn.preprocessing import MultiLabelBinarizer
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

In [2]:
## Charger le dataset nettoyé
df = pd.read_csv("../data/processed/data_clean.csv")

In [4]:
## Transformer les string des genres en liste python
df["genre"] = df["genre"].apply(ast.literal_eval)

In [ ]:
## Vérif du type de "genre"
print(type(df["genre"][0]))
df["genre"].iloc[0]

In [ ]:
genres = sorted(
    set(genre for genres in df["genre"] for genre in genres)
)

genres

In [ ]:
## Lister les genres pour la stratification
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df["genre"])
mlb.classes_

In [ ]:
## n premières lignes de la stratification des genres de livres
n = 5
y[:n]

In [ ]:
## Préparer le split du dataset grâce au msss pour test
msss = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.1,
    random_state=67
)

In [ ]:
## Spliter temporairement le dataset (90% train_tmp, 10% test)
for train_tmp_idx, test_idx in msss.split(df, y):
    train_tmp = df.iloc[train_tmp_idx].copy()
    test = df.iloc[test_idx].copy()
y_train_tmp = y[train_tmp_idx]

In [ ]:
## Vérification premier split
print("train_tmp :", train_tmp.shape)
print("test :", test.shape)

In [28]:
## Préparer le split du dataset grâce au msss pour validation
msss = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=1/9,
    random_state=67
)

In [29]:
## Spliter le dataset (80% train, 10% validation)

for train_idx, validation_idx in msss.split(train_tmp, y_train_tmp):
    train = train_tmp.iloc[train_idx].copy()
    validation = train_tmp.iloc[validation_idx].copy()

In [ ]:
## Vérification deuxième split
print("train :", train.shape)
print("validation :", validation.shape)

In [ ]:
## Vérifier découpage complet
print("Dataset complet :", df.shape)
print("Train :", train.shape)
print("Test :", test.shape)
print("Validation :", validation.shape)

In [ ]:
## Verification intersection entre jeux de données
print("Train/Test :", len(set(train["summary"]) & set(test["summary"])))
print("Train/Validation :", len(set(train["summary"]) & set(validation["summary"])))
print("Validation/Test :", len(set(validation["summary"]) & set(test["summary"])))

In [ ]:
## Vérification perte de données
print(len(train) + len(validation) + len(test))
print(len(df))

In [ ]:
## Vérification de la répartition des genres au sein des datasets
def genre_distribution(data):
    return (
        data["genre"]
        .explode()
        .value_counts(normalize=True)
        .sort_index()
        * 100
    )
print("DATASET COMPLET")
print(genre_distribution(df))

print("\nTRAIN")
print(genre_distribution(train))

print("\nVALIDATION")
print(genre_distribution(validation))

print("\nTEST")
print(genre_distribution(test))

In [36]:
train.to_csv("../data/processed/train.csv", index= False)
validation.to_csv("../data/processed/validation.csv", index= False)
test.to_csv("../data/processed/test.csv", index= False)